# Self-play evaluation

Load the committed training metrics for the large and small runs, plot checkpoint-level cross-entropy, R², and MSE, then compare each final checkpoint with greedy play. The paired greedy evaluation uses 200 shared states and a 10 × 10 × 4 throw grid; it completed in about 27 seconds in the current environment.

Training metrics are measured on each checkpoint’s retained `X_i` training data, so they describe fit to training data rather than held-out performance. Training and evaluation results are written to each run’s `stats/` folder.


In [ ]:
import json
import sys
from pathlib import Path
from time import perf_counter

repo_root = Path.cwd().resolve()
while not (repo_root / 'data_generation.py').exists():
    if repo_root.parent == repo_root:
        raise RuntimeError('Could not find repository root')
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

import matplotlib.pyplot as plt
import pandas as pd
import self_play_training as self_play


In [ ]:
weights_root = repo_root / 'weights' / 'self_play'
runs = {
    'large_m30': {'directory': weights_root / 'round9_20k', 'checkpoint': 30},
    'small_m10': {'directory': weights_root / 'round9_2k_v2', 'checkpoint': 10},
}

# This size keeps the full paired evaluation comfortably below three minutes.
seed = 2026
num_states = 200  # 100 paired states for each model-turn group
grid_size = (10, 10, 4)
max_stones = 10


In [ ]:
metric_rows = []
for run_name, run in runs.items():
    metrics_path = run['directory'] / 'stats' / 'training_metrics.json'
    metrics = json.loads(metrics_path.read_text())
    for checkpoint, values in metrics['checkpoints'].items():
        metric_rows.append({
            'run': run_name,
            'checkpoint': int(checkpoint),
            'training_rows': values.get('training_rows'),
            'cross_entropy': values.get('cross_entropy'),
            'r_squared': values.get('r_squared'),
            'mse': values.get('mse'),
        })
training_metrics = pd.DataFrame(metric_rows).sort_values(
    ['run', 'checkpoint']
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for metric, axis, label in zip(('cross_entropy', 'r_squared', 'mse'), axes, ('Cross-entropy', 'R²', 'Mean squared error')):
    for run_name, rows in training_metrics.groupby('run'):
        axis.plot(rows['checkpoint'], rows[metric], marker='o', label=run_name)
    axis.set_xlabel("Self-play checkpoint")
    axis.set_ylabel(label)
    axis.grid(alpha=0.25)
axes[0].legend()
fig.tight_layout()
plt.show()
training_metrics.tail()


In [ ]:
checkpoint_paths = {
    run_name: run["directory"] / "checkpoints" / f"m_{run['checkpoint']}.npz"
    for run_name, run in runs.items()
}
started_at = perf_counter()
greedy_results = self_play.evaluate_checkpoints_against_greedy(
    checkpoint_paths=checkpoint_paths,
    num_states=num_states,
    seed=seed,
    grid_size=grid_size,
    max_stones=max_stones,
)
elapsed_seconds = perf_counter() - started_at
print(f"Paired greedy evaluation completed in {elapsed_seconds:.1f} seconds")

for run_name, run in runs.items():
    output_dir = run['directory'] / 'stats'
    output_dir.mkdir(parents=True, exist_ok=True)
    run_results = [row for row in greedy_results if row['run'] == run_name]
    payload = {
        'run': run_name,
        'checkpoint': f"m_{run['checkpoint']}.npz",
        'parameters': {'num_states': num_states, 'seed': seed, 'grid_size': grid_size, 'max_stones': max_stones},
        'elapsed_seconds_for_all_runs': elapsed_seconds,
        'results': run_results,
    }
    (output_dir / 'greedy_comparison.json').write_text(json.dumps(payload, indent=2))


In [ ]:
greedy_summary = pd.DataFrame(greedy_results)[[
    'run', 'group', 'num_states', 'average_model_score',
    'average_greedy_score', 'average_advantage',
    'advantage_ci95_low', 'advantage_ci95_high',
    'win_rate', 'tie_rate', 'loss_rate',
]]
greedy_summary
